# 03.1 — Naive Bayes for Text Classification

**Problem it solves:** Assign a category to text (spam/not-spam, sentiment, topic) using probability theory.

**Core idea:** Apply Bayes' theorem. P(class|text) ∝ P(text|class) × P(class). The 'naive' part: assume all words are conditionally independent given the class. This is wrong but works surprisingly well.

**Why it's still relevant:** Fast, interpretable, works on tiny datasets, strong baseline.

---

In [ ]:
import math
import re
from collections import Counter, defaultdict

def tokenize(text): return re.findall(r'\b[a-z]+\b', text.lower())

class MultinomialNaiveBayes:
    """
    Multinomial Naive Bayes: models term frequency per class.
    Best for text classification (bag-of-words features).
    
    P(class|doc) ∝ P(class) × ∏_i P(word_i | class)
    In log space: log P(class) + Σ_i log P(word_i | class)
    """
    
    def __init__(self, alpha: float = 1.0):
        """
        alpha: Laplace smoothing parameter.
        alpha=1: Laplace (add-one) smoothing.
        alpha→0: maximum likelihood (no smoothing, zero probs for unseen words).
        """
        self.alpha = alpha
        self.classes = []
        self.class_log_prior: dict = {}       # log P(class)
        self.word_log_likelihood: dict = {}   # log P(word | class)
        self.vocab: set = set()
        self.class_word_counts: dict = {}     # raw counts for inspection
    
    def fit(self, texts: list[str], labels: list[str]):
        self.classes = list(set(labels))
        n_docs = len(texts)
        
        # Group documents by class
        class_docs = defaultdict(list)
        for text, label in zip(texts, labels):
            class_docs[label].append(tokenize(text))
        
        # Build vocab
        for tokens_list in class_docs.values():
            for tokens in tokens_list:
                self.vocab.update(tokens)
        V = len(self.vocab)
        
        for cls in self.classes:
            docs = class_docs[cls]
            
            # Class prior: P(class) = count(docs in class) / total docs
            self.class_log_prior[cls] = math.log(len(docs) / n_docs)
            
            # Word counts across all docs in class
            word_counts = Counter()
            for tokens in docs:
                word_counts.update(tokens)
            self.class_word_counts[cls] = word_counts
            
            total_words = sum(word_counts.values())
            
            # P(word | class) with Laplace smoothing
            # P(w|c) = (count(w,c) + alpha) / (total_words_in_c + alpha * V)
            self.word_log_likelihood[cls] = {
                word: math.log((word_counts.get(word, 0) + self.alpha) / 
                               (total_words + self.alpha * V))
                for word in self.vocab
            }
            # Store log prob for unseen words (OOV during test)
            self.word_log_likelihood[cls]['<UNK>'] = math.log(
                self.alpha / (total_words + self.alpha * V)
            )
        
        return self
    
    def _score(self, tokens: list[str], cls: str) -> float:
        """Log probability of tokens given class."""
        log_prob = self.class_log_prior[cls]
        ll = self.word_log_likelihood[cls]
        for token in tokens:
            log_prob += ll.get(token, ll['<UNK>'])
        return log_prob
    
    def predict(self, text: str) -> str:
        tokens = tokenize(text)
        scores = {cls: self._score(tokens, cls) for cls in self.classes}
        return max(scores, key=scores.get)
    
    def predict_proba(self, text: str) -> dict:
        tokens = tokenize(text)
        log_scores = {cls: self._score(tokens, cls) for cls in self.classes}
        # Convert log scores to probabilities via softmax
        max_log = max(log_scores.values())
        exp_scores = {cls: math.exp(s - max_log) for cls, s in log_scores.items()}
        total = sum(exp_scores.values())
        return {cls: v/total for cls, v in exp_scores.items()}
    
    def top_features(self, cls: str, n: int = 10) -> list:
        """Words most indicative of a class (by log-likelihood ratio)."""
        ll = self.word_log_likelihood[cls]
        return sorted(ll.items(), key=lambda x: x[1], reverse=True)[:n]

# Sentiment classification
train_data = [
    ("this movie is great and wonderful", "positive"),
    ("amazing film loved every minute", "positive"),
    ("fantastic performances brilliant screenplay", "positive"),
    ("incredible story beautiful cinematography loved it", "positive"),
    ("best movie I have seen in years highly recommend", "positive"),
    ("this film is terrible boring and slow", "negative"),
    ("awful acting dreadful script waste of time", "negative"),
    ("disappointing and dull not worth watching", "negative"),
    ("horrible movie very bad acting poor direction", "negative"),
    ("worst film ever made complete garbage", "negative"),
]
texts, labels = zip(*train_data)

nb = MultinomialNaiveBayes(alpha=1.0)
nb.fit(list(texts), list(labels))

test_cases = [
    "wonderful performances loved the story",
    "terrible boring awful waste of time",
    "decent movie some good scenes some bad ones",  # ambiguous
    "this is the greatest worst film ever",          # tricky
]

print("Predictions:")
for text in test_cases:
    pred = nb.predict(text)
    proba = nb.predict_proba(text)
    print(f"  {text!r}")
    print(f"    -> {pred}  (P(pos)={proba['positive']:.3f}, P(neg)={proba['negative']:.3f})")

print("\nTop positive words:")
for word, log_p in nb.top_features('positive', 8):
    print(f"  {word:20} log_p={log_p:.3f}")

In [ ]:
# Why does Laplace smoothing matter?

# Without smoothing (alpha=0), unseen words have P=0
# One unseen word in a document -> P(class|doc) = 0 forever
# This is called the 'zero frequency problem'

nb_no_smooth = MultinomialNaiveBayes(alpha=0.001)  # near-zero smoothing
nb_no_smooth.fit(list(texts), list(labels))

nb_smooth = MultinomialNaiveBayes(alpha=1.0)
nb_smooth.fit(list(texts), list(labels))

unseen_word_test = "xenomorph fantastic incredible"
print(f"Test: {unseen_word_test!r}")
print(f"  No smoothing (alpha=0.001): {nb_no_smooth.predict(unseen_word_test)} {nb_no_smooth.predict_proba(unseen_word_test)}")
print(f"  Laplace (alpha=1.0):        {nb_smooth.predict(unseen_word_test)} {nb_smooth.predict_proba(unseen_word_test)}")
print()
print("With no smoothing, 'xenomorph' (unseen) → P(word|class) ≈ 0 → log(0) = -inf")
print("Smoothing prevents this by adding pseudocounts")

In [ ]:
# NB vs Logistic Regression: the 'generative vs discriminative' distinction
# NB models P(x|y) — the data-generating process
# LR models P(y|x) directly — the decision boundary
# NB is better with LESS data; LR catches up with more data

# Cross-validation to measure accuracy
def cross_validate(model_class, texts, labels, n_folds=5):
    fold_size = len(texts) // n_folds
    accuracies = []
    for i in range(n_folds):
        start, end = i * fold_size, (i + 1) * fold_size
        test_x = texts[start:end]
        test_y = labels[start:end]
        train_x = texts[:start] + texts[end:]
        train_y = labels[:start] + labels[end:]
        model = model_class().fit(train_x, train_y)
        preds = [model.predict(t) for t in test_x]
        acc = sum(p == g for p, g in zip(preds, test_y)) / len(test_y)
        accuracies.append(acc)
    return sum(accuracies) / len(accuracies)

all_texts = list(texts) * 3  # replicate to have more data
all_labels = list(labels) * 3

acc = cross_validate(MultinomialNaiveBayes, all_texts, all_labels, n_folds=5)
print(f"NB 5-fold CV accuracy: {acc:.3f}")

## Summary

**Naive Bayes works because:** Even though word independence is wrong, the decision boundary is often right — the errors cancel out.

**When to use:** Fast baseline, small data, interpretable features, real-time classification without GPU.

**When NB fails:**
- Strong word correlations (e.g., 'not good' — 'not' and 'good' aren't independent)
- Rare but important words get washed out by smoothing
- Requires balanced classes or adjusted priors

---
**Next:** 03.2 — Logistic Regression for text